Importing Needed Libraries - Routing to Proper Device

In [33]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader, random_split, Dataset
from PIL import Image

print("torch:", torch.__version__)
device = "cuda"  # Change device in the runtime settings - to GPU T4
print("device:", device)

def set_seed(seed: int = 42):                                                                       # Here for reproducability
    """Make results as reproducible as possible across runs."""
    import os, random
    import numpy as np
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Deterministic flags (safe on CPU; on GPU some ops may error if non-deterministic)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:
        print("Warning: could not enable full deterministic algorithms:", e)

set_seed(42)                                                                                        # Here for reproducability


torch: 2.11.0+cpu
device: cuda


Importing From Kaggle API - Get Kaggle key first - Already Downloaded

In [23]:
# Code that kaggle said to run to get access to the competition files
import kagglehub

# Download latest version
path = kagglehub.competition_download('ucsc-cse-144-spring-2026-final-project')

print("Path to competition files:", path)

100%|██████████| 120M/120M [00:01<00:00, 92.3MB/s]

Extracting files...


Path to competition files: /root/.cache/kagglehub/competitions/ucsc-cse-144-spring-2026-final-project


Data Pipeline - Dataset and DataLoader - IN PROGRESS

In [36]:

batch_size = 64  # For now, can change to 64?
num_workers = 0                                                                 # Here for reproducability
mean=[0.485, 0.456, 0.406]                                                      # ImageNet stats - that is what the model was trained on
std=[0.229, 0.224, 0.225]
val_p = 0.2 # portion of training data


class UnlabeledDataset(Dataset):
  def __init__(self, root, transform=None):
      self.root   = root
      self.transform = transform
      self.images    = sorted(os.listdir(root))

  def __len__(self):
      return len(self.images)

  def __getitem__(self, idx):
      img_path = os.path.join(self.root, self.images[idx])
      image    = Image.open(img_path).convert('RGB')
      if self.transform:
          image = self.transform(image)
      return image, self.images[idx]

train_path = path + '/train'
test_path = path + '/test'

train_tf = v2.Compose([
           v2.ToImage(),
           v2.ToDtype(torch.uint8, scale=True),
           v2.Resize((224, 224)),
           v2.RandomResizedCrop(size=(224, 224), antialias=True),   # Augmentation
           v2.RandomHorizontalFlip(p=0.5),                          # Augmentation
           v2.ToDtype(torch.float32, scale=True),
           v2.Normalize(mean, std)
])

test_tf = v2.Compose([
          v2.ToImage(),
          v2.Resize((224, 224)),
          v2.Normalize(mean, std)
])

train_set = datasets.ImageFolder(root=train_path, transform=train_tf)
val_set   = datasets.ImageFolder(root=train_path,  transform=test_tf)
test_set   = UnlabeledDataset(root=test_path,  transform=test_tf)

train_size = int((1-val_p) * len(train_set))
val_size   = len(train_set) - train_size

train_set, val_set = random_split(train_set, [train_size, val_size])

train_loader = DataLoader(dataset=train_set,
                          batch_size=64,
                          shuffle=True,
                          num_workers=0
)
val_loader =   DataLoader(dataset=val_set,
                          batch_size=64,
                          shuffle=False,
                          num_workers=0
)
test_loader =  DataLoader(dataset=test_set,
                          batch_size=64,
                          shuffle=False,
                          num_workers=0
)

print("train/val/test:", len(train_set), len(val_set), len(test_set))

train/val/test: 863 216 1036
